In [133]:
import importlib
import src.audit.quality as quality
import src.audit.match_integrity as integrity
import src.audit.schema as schema
import src.audit.team_coverage as team_coverage
import src.audit.anomalies as anomalies

importlib.reload(quality)
importlib.reload(integrity)
importlib.reload(schema)
importlib.reload(team_coverage)
importlib.reload(anomalies)

print([name for name in dir(quality) if not name.startswith("_")])

['audit_categorical_domains', 'audit_duplicates', 'audit_missingness', 'audit_value_domains', 'pd']


In [134]:
from pathlib import Path
import sys
import pandas as pd
from src.audit.schema import audit_schema
from src.audit.quality import audit_missingness, audit_duplicates, audit_value_domains, audit_categorical_domains
from src.audit.match_integrity import audit_match_metadata_consistency, audit_match_innings_structure, audit_innings_length, audit_match_results
from src.audit.team_coverage import audit_team_coverage
from src.audit.anomalies import (
    audit_team_name_consistency,
    audit_innings_length_anomalies,
    audit_score_extras_reconciliation,
    audit_wicket_semantics,
    audit_tactical_missingness,
    audit_delivery_key,
    audit_incomplete_matches,
    audit_schema_naming,
    audit_delivery_order
)

In [ ]:
PROJECT_ROOT = Path.cwd().parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.data_loader import load_raw_data

df = load_raw_data()

print("Shape:", df.shape)

Shape: (34967, 61)


In [ ]:
df[["p_match", "inns", "over_num", "ball", "ball_id"]].head(20)

,p_match,inns,over_num,ball,ball_id
0,1359475,1,1,1,0.01
1,1359475,1,1,2,0.02
2,1359475,1,1,3,0.03
3,1359475,1,1,4,0.04
4,1359475,1,1,5,0.05
5,1359475,1,1,6,0.06
6,1359475,1,2,1,1.01
7,1359475,1,2,2,1.02
8,1359475,1,2,3,1.03
9,1359475,1,2,4,1.04


In [ ]:
df.groupby(["p_match", "inns"]).size().describe()

count    289.000000
mean     120.993080
std       11.038441
min       55.000000
25%      121.000000
50%      124.000000
75%      127.000000
max      136.000000
dtype: float64

In [ ]:
df.duplicated(
    subset=["p_match", "inns", "ball_id"]
).sum()

np.int64(0)

In [ ]:
schema_audit = audit_schema(df)

schema_audit

,column,dtype,non_null,missing,missing_pct,unique
0,row_id,int64,34967,0,0.000000,34967
1,p_match,int64,34967,0,0.000000,145
2,inns,int64,34967,0,0.000000,2
3,bat,object,34967,0,0.000000,232
4,p_bat,int64,34967,0,0.000000,232
...,...,...,...,...,...,...
56,predscore,int64,34967,0,0.000000,219
57,wprob,float64,34967,0,0.000000,8529
58,bowl_speed_category,float64,32588,2379,6.803558,8
59,bowl_release_point,float64,32588,2379,6.803558,4


In [ ]:
print(f"Number of rows   : {df.shape[0]:,}")
print(f"Number of columns: {df.shape[1]}")
print(f"Duplicate rows   : {df.duplicated().sum():,}")

Number of rows   : 34,967
Number of columns: 61
Duplicate rows   : 0


In [ ]:
missingness_audit = audit_missingness(df)

missingness_audit

,column,missing,missing_pct
0,dismissal,33168,94.855149
1,target,18136,51.866045
2,inns_runs_rem,18136,51.866045
3,inns_rrr,18136,51.866045
4,bowl_release_point,2379,6.803558
...,...,...,...
56,cur_bowl_runs,0,0.000000
57,inns_runs,0,0.000000
58,inns_wkts,0,0.000000
59,inns_balls,0,0.000000


In [ ]:
# Show columns with missing values
missingness_audit[
    missingness_audit["missing"] > 0
]

,column,missing,missing_pct
0,dismissal,33168,94.855149
1,target,18136,51.866045
2,inns_runs_rem,18136,51.866045
3,inns_rrr,18136,51.866045
4,bowl_release_point,2379,6.803558
5,bowl_speed_category,2379,6.803558
6,bowl_type,1936,5.536649
7,control,963,2.754025
8,shot,194,0.554809
9,line,56,0.160151


In [ ]:
# Show the highest-missing columns
missingness_audit[
    missingness_audit["missing_pct"] > 1
]

,column,missing,missing_pct
0,dismissal,33168,94.855149
1,target,18136,51.866045
2,inns_runs_rem,18136,51.866045
3,inns_rrr,18136,51.866045
4,bowl_release_point,2379,6.803558
5,bowl_speed_category,2379,6.803558
6,bowl_type,1936,5.536649
7,control,963,2.754025


In [ ]:
second_innings_fields = [
    "target",
    "inns_runs_rem",
    "inns_rrr"
]

df.groupby("inns")[second_innings_fields].apply(
    lambda x: x.notna().mean() * 100
)

,target,inns_runs_rem,inns_rrr
inns,,,
1,0.0,0.0,0.0
2,100.0,100.0,100.0


In [30]:
duplicate_audit = audit_duplicates(df)

duplicate_audit

{'duplicate_rows': np.int64(0),
 'duplicate_row_id': np.int64(0),
 'duplicate_delivery_key': np.int64(0)}

In [31]:
print("Missing row_id:", df["row_id"].isna().sum())

delivery_key_columns = ["p_match", "inns", "ball_id"]

print("\nMissing delivery-key values:")
print(df[delivery_key_columns].isna().sum())

Missing row_id: 0

Missing delivery-key values:
p_match    0
inns       0
ball_id    0
dtype: int64


In [37]:
value_domain_audit = audit_value_domains(df)

value_domain_audit

{'innings_values': [1, 2],
 'year_values': [2023, 2024],
 'competition_values': ['IPL'],
 'out_values': [False, True],
 'bat_out_values': [False, True],
 'noball_values': [0, 1],
 'wide_values': [0, 1, 2, 3, 5],
 'byes_values': [0, 1, 2, 4],
 'legbyes_values': [0, 1, 2, 4]}

In [40]:
categorical_domain_audit = audit_categorical_domains(df)

categorical_domain_audit

{'team_bat': ['Chennai Super Kings',
  'Delhi Capitals',
  'Gujarat Titans',
  'Kolkata Knight Riders',
  'Lucknow Super Giants',
  'Mumbai Indians',
  'Punjab Kings',
  'Rajasthan Royals',
  'Royal Challengers Bengaluru',
  'Sunrisers Hyderabad'],
 'team_bowl': ['Chennai Super Kings',
  'Delhi Capitals',
  'Gujarat Titans',
  'Kolkata Knight Riders',
  'Lucknow Super Giants',
  'Mumbai Indians',
  'Punjab Kings',
  'Rajasthan Royals',
  'Royal Challengers Bengaluru',
  'Sunrisers Hyderabad'],
 'winner': ['-',
  'Chennai Super Kings',
  'Delhi Capitals',
  'Gujarat Titans',
  'Kolkata Knight Riders',
  'Lucknow Super Giants',
  'Mumbai Indians',
  'Punjab Kings',
  'Rajasthan Royals',
  'Royal Challengers Bangalore',
  'Royal Challengers Bengaluru',
  'Sunrisers Hyderabad'],
 'toss': ['Chennai Super Kings',
  'Delhi Capitals',
  'Gujarat Titans',
  'Kolkata Knight Riders',
  'Lucknow Super Giants',
  'Mumbai Indians',
  'Punjab Kings',
  'Rajasthan Royals',
  'Royal Challengers Bangalo

In [47]:
match_metadata_audit = audit_match_metadata_consistency(df)

match_metadata_audit.head()

,p_match,year,match_date,ground,country,winner,toss,competition
0,1359475,1,1,1,1,1,1,1
1,1359476,1,1,1,1,1,1,1
2,1359477,1,1,1,1,1,1,1
3,1359478,1,1,1,1,1,1,1
4,1359479,1,1,1,1,1,1,1


In [48]:
metadata_columns = [
    "year",
    "match_date",
    "ground",
    "country",
    "winner",
    "toss",
    "competition",
]

problematic_matches = match_metadata_audit[
    match_metadata_audit[metadata_columns].gt(1).any(axis=1)
]

problematic_matches

,p_match,year,match_date,ground,country,winner,toss,competition


In [45]:
print("Total matches:", match_metadata_audit["p_match"].nunique())
print("Matches with metadata inconsistencies:", len(problematic_matches))

Total matches: 145
Matches with metadata inconsistencies: 0


In [46]:
rcb_matches = df[
    df["winner"].isin([
        "Royal Challengers Bangalore",
        "Royal Challengers Bengaluru"
    ])
    | df["toss"].isin([
        "Royal Challengers Bangalore",
        "Royal Challengers Bengaluru"
    ])
]

rcb_matches.groupby("p_match")[["winner", "toss"]].first()

,winner,toss
p_match,,
1359479,Royal Challengers Bangalore,Royal Challengers Bangalore
1359483,Kolkata Knight Riders,Royal Challengers Bangalore
1359494,Royal Challengers Bangalore,Delhi Capitals
1359498,Chennai Super Kings,Royal Challengers Bangalore
1359501,Royal Challengers Bangalore,Punjab Kings
1359506,Royal Challengers Bangalore,Rajasthan Royals
1359510,Kolkata Knight Riders,Royal Challengers Bangalore
1359517,Royal Challengers Bangalore,Royal Challengers Bangalore
1359524,Delhi Capitals,Royal Challengers Bangalore


In [52]:
innings_structure_audit = audit_match_innings_structure(df)

innings_structure_audit.head()

,p_match,innings_count,total_deliveries,max_balls_in_match
0,1359475,2,243,120
1,1359476,2,222,120
2,1359477,2,251,120
3,1359478,2,248,120
4,1359479,2,232,120


In [53]:
innings_structure_audit["innings_count"].value_counts().sort_index()

innings_count
1      1
2    144
Name: count, dtype: int64

In [54]:
unusual_innings_matches = innings_structure_audit[
    innings_structure_audit["innings_count"] != 2
]

unusual_innings_matches

,p_match,innings_count,total_deliveries,max_balls_in_match
44,1359519,1,119,120


In [57]:
innings_length_audit = audit_innings_length(df)

innings_length_audit.head()

,p_match,inns,max_balls,actual_deliveries
0,1359475,1,120,121
1,1359475,2,120,122
2,1359476,1,0,123
3,1359476,2,120,99
4,1359477,1,120,125


In [58]:
innings_length_audit["max_balls"].value_counts().sort_index()

max_balls
0        2
90       1
96       2
120    284
Name: count, dtype: int64

In [59]:
shortened_innings = innings_length_audit[
    innings_length_audit["max_balls"] < 120
]

shortened_innings

,p_match,inns,max_balls,actual_deliveries
2,1359476,1,0,123
145,1370353,1,0,122
146,1370353,2,90,94
265,1426298,1,96,99
266,1426298,2,96,102


In [60]:
df.groupby(["p_match", "inns"]).agg(
    max_balls=("max_balls", "first"),
    deliveries=("ball_id", "count"),
    legal_balls=("wide", lambda x: (x == 0).sum())
).reset_index().query("max_balls < 120")

,p_match,inns,max_balls,deliveries,legal_balls
2,1359476,1,0,123,121
145,1370353,1,0,122,120
146,1370353,2,90,94,90
265,1426298,1,96,99,96
266,1426298,2,96,102,97


In [61]:
df.groupby("p_match").agg(
    winner=("winner", "first"),
    teams=("team_bat", lambda x: sorted(x.unique().tolist())),
    innings=("inns", "nunique")
).reset_index().head(10)

,p_match,winner,teams,innings
0,1359475,Gujarat Titans,"[Chennai Super Kings, Gujarat Titans]",2
1,1359476,Punjab Kings,"[Kolkata Knight Riders, Punjab Kings]",2
2,1359477,Lucknow Super Giants,"[Delhi Capitals, Lucknow Super Giants]",2
3,1359478,Rajasthan Royals,"[Rajasthan Royals, Sunrisers Hyderabad]",2
4,1359479,Royal Challengers Bangalore,"[Mumbai Indians, Royal Challengers Bengaluru]",2
5,1359480,Chennai Super Kings,"[Chennai Super Kings, Lucknow Super Giants]",2
6,1359481,Gujarat Titans,"[Delhi Capitals, Gujarat Titans]",2
7,1359482,Punjab Kings,"[Punjab Kings, Rajasthan Royals]",2
8,1359483,Kolkata Knight Riders,"[Kolkata Knight Riders, Royal Challengers Beng...",2
9,1359484,Lucknow Super Giants,"[Lucknow Super Giants, Sunrisers Hyderabad]",2


In [62]:
match_results = (
    df.groupby("p_match")
      .agg(
          winner=("winner", "first"),
          innings=("inns", "nunique"),
          teams=("team_bat", lambda x: sorted(x.unique().tolist()))
      )
      .reset_index()
)

match_results["winner"].value_counts(dropna=False)

winner
Chennai Super Kings            17
Kolkata Knight Riders          17
Gujarat Titans                 16
Rajasthan Royals               16
Lucknow Super Giants           15
Sunrisers Hyderabad            13
Mumbai Indians                 13
Delhi Capitals                 12
Punjab Kings                   11
Royal Challengers Bangalore     7
Royal Challengers Bengaluru     7
-                               1
Name: count, dtype: int64

In [63]:
no_winner_matches = match_results[
    match_results["winner"].isna() |
    (match_results["winner"] == "-")
]

no_winner_matches

,p_match,winner,innings,teams
44,1359519,-,1,[Lucknow Super Giants]


In [64]:
match_results["winner_in_teams"] = match_results.apply(
    lambda row: (
        row["winner"] == "-"
        or row["winner"] in row["teams"]
    ),
    axis=1
)

match_results[
    ~match_results["winner_in_teams"]
]

,p_match,winner,innings,teams,winner_in_teams
4,1359479,Royal Challengers Bangalore,2,"[Mumbai Indians, Royal Challengers Bengaluru]",False
19,1359494,Royal Challengers Bangalore,2,"[Delhi Capitals, Royal Challengers Bengaluru]",False
26,1359501,Royal Challengers Bangalore,2,"[Punjab Kings, Royal Challengers Bengaluru]",False
31,1359506,Royal Challengers Bangalore,2,"[Rajasthan Royals, Royal Challengers Bengaluru]",False
42,1359517,Royal Challengers Bangalore,2,"[Lucknow Super Giants, Royal Challengers Benga...",False
59,1359534,Royal Challengers Bangalore,2,"[Rajasthan Royals, Royal Challengers Bengaluru]",False
64,1359539,Royal Challengers Bangalore,2,"[Royal Challengers Bengaluru, Sunrisers Hydera...",False


In [65]:
print(
    "Matches where winner is not one of the participating teams:",
    (~match_results["winner_in_teams"]).sum()
)

Matches where winner is not one of the participating teams: 7


In [66]:
innings_per_match = (
    df.groupby("p_match")["inns"]
      .unique()
      .reset_index()
)

innings_per_match.head()

,p_match,inns
0,1359475,"[1, 2]"
1,1359476,"[1, 2]"
2,1359477,"[1, 2]"
3,1359478,"[1, 2]"
4,1359479,"[1, 2]"


In [67]:
invalid_innings_sequences = innings_per_match[
    ~innings_per_match["inns"].apply(
        lambda x: list(x) == [1, 2] or list(x) == [1]
    )
]

invalid_innings_sequences

,p_match,inns
11,1359486,"[2, 1]"


In [68]:
ball_id_summary = (
    df.groupby(["p_match", "inns"])
      .agg(
          first_ball_id=("ball_id", "min"),
          last_ball_id=("ball_id", "max"),
          delivery_rows=("ball_id", "count"),
          unique_ball_ids=("ball_id", "nunique")
      )
      .reset_index()
)

ball_id_summary[
    ball_id_summary["delivery_rows"] != ball_id_summary["unique_ball_ids"]
]

,p_match,inns,first_ball_id,last_ball_id,delivery_rows,unique_ball_ids


In [69]:
over_summary = (
    df.groupby(["p_match", "inns"])
      .agg(
          first_over=("over_num", "min"),
          last_over=("over_num", "max"),
          unique_overs=("over_num", "nunique")
      )
      .reset_index()
)

over_summary.head()

,p_match,inns,first_over,last_over,unique_overs
0,1359475,1,1,20,20
1,1359475,2,1,20,20
2,1359476,1,1,20,20
3,1359476,2,1,16,16
4,1359477,1,1,20,20


In [70]:
id="x5g4ny"
df[df["p_match"] == 1359486][
    ["inns", "ball_id", "over_num", "team_bat", "team_bowl"]
].head(20)

,inns,ball_id,over_num,team_bat,team_bowl
2006,2,4.05,5,Chennai Super Kings,Mumbai Indians
2667,1,0.01,1,Mumbai Indians,Chennai Super Kings
2668,1,0.02,1,Mumbai Indians,Chennai Super Kings
2669,1,0.03,1,Mumbai Indians,Chennai Super Kings
2670,1,0.04,1,Mumbai Indians,Chennai Super Kings
2671,1,0.05,1,Mumbai Indians,Chennai Super Kings
2672,1,0.06,1,Mumbai Indians,Chennai Super Kings
2673,1,1.01,2,Mumbai Indians,Chennai Super Kings
2674,1,1.02,2,Mumbai Indians,Chennai Super Kings
2675,1,1.03,2,Mumbai Indians,Chennai Super Kings


In [71]:
id="l6w1cg"
df[df["p_match"] == 1359486].groupby("inns").agg(
    first_ball_id=("ball_id", "min"),
    last_ball_id=("ball_id", "max"),
    first_over=("over_num", "min"),
    last_over=("over_num", "max"),
    rows=("ball_id", "count"),
    team_bat=("team_bat", "first"),
    team_bowl=("team_bowl", "first")
)

,first_ball_id,last_ball_id,first_over,last_over,rows,team_bat,team_bowl
inns,,,,,,,
1,0.01,19.08,1,20,125,Mumbai Indians,Chennai Super Kings
2,0.01,18.02,1,19,114,Chennai Super Kings,Mumbai Indians


In [72]:
score_anomalies = df[
    df["score"] != (
        df["batruns"]
        + df["wide"]
        + df["noball"]
        + df["byes"]
        + df["legbyes"]
    )
]

score_anomalies[
    [
        "p_match",
        "inns",
        "ball_id",
        "outcome",
        "score",
        "batruns",
        "wide",
        "noball",
        "byes",
        "legbyes",
    ]
]

,p_match,inns,ball_id,outcome,score,batruns,wide,noball,byes,legbyes
1452,1359480,2,19.02,leg bye,1,0,0,1,0,1
1813,1359482,1,18.04,bye,1,0,0,1,1,0
4565,1359493,2,9.04,leg bye,1,0,0,1,0,1
4951,1359495,1,10.05,leg bye,1,0,0,1,0,4
9151,1359512,1,15.05,leg bye,1,0,0,1,0,1
13024,1359528,2,12.03,leg bye,1,0,0,1,0,4
15180,1359537,2,18.04,bye,1,0,0,1,4,0
17044,1370350,1,19.04,bye,1,0,0,1,1,0
20870,1422131,1,5.02,leg bye,1,0,0,1,0,1
21786,1422134,2,14.03,bye,1,0,0,1,4,0


In [73]:
score_anomalies["outcome"].value_counts()

outcome
leg bye    8
bye        6
Name: count, dtype: int64

In [74]:
score_anomalies[
    [
        "score",
        "batruns",
        "wide",
        "noball",
        "byes",
        "legbyes"
    ]
]

,score,batruns,wide,noball,byes,legbyes
1452,1,0,0,1,0,1
1813,1,0,0,1,1,0
4565,1,0,0,1,0,1
4951,1,0,0,1,0,4
9151,1,0,0,1,0,1
13024,1,0,0,1,0,4
15180,1,0,0,1,4,0
17044,1,0,0,1,1,0
20870,1,0,0,1,0,1
21786,1,0,0,1,4,0


In [75]:
wicket_summary = (
    df.groupby("out")
      .agg(
          rows=("row_id", "count"),
          dismissal_populated=("dismissal", lambda x: x.notna().sum()),
          p_out_populated=("p_out", lambda x: x.notna().sum()),
          bat_out_true=("bat_out", "sum")
      )
      .reset_index()
)

wicket_summary

,out,rows,dismissal_populated,p_out_populated,bat_out_true
0,False,33168,0,33168,33168
1,True,1799,1799,1799,1747


In [79]:
dismissal_vs_out = pd.crosstab(
    df["out"],
    df["dismissal"].notna(),
    margins=True
)

dismissal_vs_out

dismissal,False,True,All
out,,,
False,33168,0,33168
True,0,1799,1799
All,33168,1799,34967


In [80]:
df[df["out"] == True][
    [
        "p_match",
        "inns",
        "ball_id",
        "p_bat",
        "p_out",
        "out",
        "bat_out",
        "dismissal"
    ]
].head(30)

,p_match,inns,ball_id,p_bat,p_out,out,bat_out,dismissal
13,1359475,1,2.02,379140,379140,True,True,bowled
35,1359475,1,5.05,8917,8917,True,True,caught
47,1359475,1,7.04,311158,311158,True,True,caught
78,1359475,1,12.05,33141,33141,True,True,bowled
104,1359475,1,17.01,1060380,1060380,True,True,caught
107,1359475,1,17.04,234675,234675,True,True,caught
112,1359475,1,18.03,714451,714451,True,True,caught
147,1359475,2,3.07,279810,279810,True,True,caught
181,1359475,2,9.03,1151288,1151288,True,True,caught
198,1359475,2,12.01,625371,625371,True,True,bowled


In [81]:
bat_out_check = df[df["out"] == True].copy()

bat_out_check["p_out_equals_p_bat"] = (
    bat_out_check["p_out"] == bat_out_check["p_bat"]
)

pd.crosstab(
    bat_out_check["bat_out"],
    bat_out_check["p_out_equals_p_bat"],
    margins=True
)

p_out_equals_p_bat,False,True,All
bat_out,,,
False,52,0,52
True,0,1747,1747
All,52,1747,1799


In [82]:
bat_out_check[
    bat_out_check["bat_out"] == False
][
    [
        "p_match",
        "inns",
        "ball_id",
        "p_bat",
        "p_out",
        "out",
        "bat_out",
        "dismissal"
    ]
].head(60)

,p_match,inns,ball_id,p_bat,p_out,out,bat_out,dismissal
1760,1359482,1,10.01,28235,342619,True,False,retired not out (hurt)
2305,1359484,1,19.01,1175485,1246528,True,False,run out
4257,1359492,1,19.07,1168641,290727,True,False,run out
5399,1359497,1,4.06,1070173,1151288,True,False,run out
5989,1359499,1,19.06,1076713,892749,True,False,run out
6237,1359500,1,19.05,471342,604302,True,False,run out
6316,1359500,2,12.04,308967,425943,True,False,run out
6521,1359501,2,5.04,1161024,340854,True,False,run out
6722,1359502,1,19.06,276298,1108375,True,False,run out
6964,1359503,1,19.07,696401,719715,True,False,run out


In [83]:
pd.crosstab(
    df["out"],
    df["p_out"].isna()
)

p_out,False
out,
False,33168
True,1799


In [84]:
wicket_rows = df[df["out"] == True]

(
    wicket_rows["p_out"].isna().sum(),
    wicket_rows["p_out"].isin(
        pd.concat([df["p_bat"], df["p_out"]]).dropna().unique()
    ).sum()
)

(np.int64(0), np.int64(1799))

In [87]:
match_result_audit = audit_match_results(df)

match_result_audit.head()

,p_match,winner,innings_count,teams,winner_in_teams,complete_two_innings
0,1359475,Gujarat Titans,2,"[Chennai Super Kings, Gujarat Titans]",True,True
1,1359476,Punjab Kings,2,"[Kolkata Knight Riders, Punjab Kings]",True,True
2,1359477,Lucknow Super Giants,2,"[Delhi Capitals, Lucknow Super Giants]",True,True
3,1359478,Rajasthan Royals,2,"[Rajasthan Royals, Sunrisers Hyderabad]",True,True
4,1359479,Royal Challengers Bangalore,2,"[Mumbai Indians, Royal Challengers Bengaluru]",False,True


In [88]:
match_result_audit["complete_two_innings"].value_counts()

complete_two_innings
True     144
False      1
Name: count, dtype: int64

In [89]:
match_result_audit["winner_in_teams"].value_counts()

winner_in_teams
True     138
False      7
Name: count, dtype: int64

In [90]:
match_result_audit[
    ~match_result_audit["winner_in_teams"]
]

,p_match,winner,innings_count,teams,winner_in_teams,complete_two_innings
4,1359479,Royal Challengers Bangalore,2,"[Mumbai Indians, Royal Challengers Bengaluru]",False,True
19,1359494,Royal Challengers Bangalore,2,"[Delhi Capitals, Royal Challengers Bengaluru]",False,True
26,1359501,Royal Challengers Bangalore,2,"[Punjab Kings, Royal Challengers Bengaluru]",False,True
31,1359506,Royal Challengers Bangalore,2,"[Rajasthan Royals, Royal Challengers Bengaluru]",False,True
42,1359517,Royal Challengers Bangalore,2,"[Lucknow Super Giants, Royal Challengers Benga...",False,True
59,1359534,Royal Challengers Bangalore,2,"[Rajasthan Royals, Royal Challengers Bengaluru]",False,True
64,1359539,Royal Challengers Bangalore,2,"[Royal Challengers Bengaluru, Sunrisers Hydera...",False,True


In [93]:
incomplete_matches = match_result_audit[~match_result_audit["complete_two_innings"]]
incomplete_matches

,p_match,winner,innings_count,teams,winner_in_teams,complete_two_innings
44,1359519,-,1,[Lucknow Super Giants],True,False


In [94]:
player_bat_identity = (
    df.groupby("p_bat")["bat"]
      .nunique()
      .sort_values(ascending=False)
)

player_bat_identity.head(20)

p_bat
8917       1
1048813    1
917231     1
926851     1
930189     1
931581     1
940973     1
942367     1
956871     1
959767     1
961713     1
974087     1
974175     1
1048739    1
1060380    1
25913      1
1064812    1
1070168    1
1070173    1
1070180    1
Name: bat, dtype: int64

In [95]:
player_bat_identity[player_bat_identity > 1]

Series([], Name: bat, dtype: int64)

In [96]:
player_bowl_identity = (
    df.groupby("p_bowl")["bowl"]
      .nunique()
      .sort_values(ascending=False)
)

player_bowl_identity[player_bowl_identity > 1]

Series([], Name: bowl, dtype: int64)

In [97]:
player_bat_teams = (
    df.groupby("p_bat")["team_bat"]
      .nunique()
      .sort_values(ascending=False)
)

player_bat_teams[player_bat_teams > 1]

p_bat
719719     2
625371     2
390484     2
390481     2
1119026    2
1076713    2
376116     2
475281     2
318845     2
1159711    2
595978     2
820351     2
1244751    2
232292     2
290630     2
669365     2
694211     2
677077     2
Name: team_bat, dtype: int64

In [98]:
player_bowl_teams = (
    df.groupby("p_bowl")["team_bowl"]
      .nunique()
      .sort_values(ascending=False)
)

player_bowl_teams[player_bowl_teams > 1]

p_bowl
1244751    2
390484     2
1159720    2
1159711    2
376116     2
670031     2
694211     2
595978     2
942367     2
390481     2
232292     2
493773     2
1076713    2
1122918    2
475281     2
330902     2
625371     2
Name: team_bowl, dtype: int64

In [99]:
bat_team_exceptions = (
    df[df["p_bat"].isin(player_bat_teams[player_bat_teams > 1].index)]
    .groupby(["p_bat", "bat"])
    .agg(
        teams=("team_bat", lambda x: sorted(x.dropna().unique().tolist())),
        years=("year", lambda x: sorted(x.dropna().unique().tolist())),
        first_date=("match_date", "min"),
        last_date=("match_date", "max"),
    )
    .reset_index()
)

bat_team_exceptions

,p_bat,bat,teams,years,first_date,last_date
0,232292,Swapnil Singh,"[Lucknow Super Giants, Royal Challengers Benga...","[2023, 2024]",4/25/2024,5/9/2024
1,290630,Manish Pandey,"[Delhi Capitals, Kolkata Knight Riders]","[2023, 2024]",4/11/2023,5/3/2024
2,318845,Rilee Rossouw,"[Delhi Capitals, Punjab Kings]","[2023, 2024]",4/1/2023,5/9/2024
3,376116,Umesh Yadav,"[Gujarat Titans, Kolkata Knight Riders]","[2023, 2024]",3/26/2024,4/9/2023
4,390481,Harshal Patel,"[Punjab Kings, Royal Challengers Bengaluru]","[2023, 2024]",4/15/2023,5/9/2024
5,390484,Jaydev Unadkat,"[Lucknow Super Giants, Sunrisers Hyderabad]","[2023, 2024]",4/10/2023,5/26/2024
6,475281,Shardul Thakur,"[Chennai Super Kings, Kolkata Knight Riders]","[2023, 2024]",4/1/2023,5/5/2024
7,595978,Tristan Stubbs,"[Delhi Capitals, Mumbai Indians]","[2023, 2024]",3/23/2024,5/7/2024
8,625371,Hardik Pandya,"[Gujarat Titans, Mumbai Indians]","[2023, 2024]",3/24/2024,5/7/2023
9,669365,Phil Salt,"[Delhi Capitals, Kolkata Knight Riders]","[2023, 2024]",3/23/2024,5/6/2023


In [100]:
bowl_team_exceptions = (
    df[df["p_bowl"].isin(player_bowl_teams[player_bowl_teams > 1].index)]
    .groupby(["p_bowl", "bowl"])
    .agg(
        teams=("team_bowl", lambda x: sorted(x.dropna().unique().tolist())),
        years=("year", lambda x: sorted(x.dropna().unique().tolist())),
        first_date=("match_date", "min"),
        last_date=("match_date", "max"),
    )
    .reset_index()
)

bowl_team_exceptions

,p_bowl,bowl,teams,years,first_date,last_date
0,232292,Swapnil Singh,"[Lucknow Super Giants, Royal Challengers Benga...","[2023, 2024]",4/25/2024,5/9/2024
1,330902,Mustafizur Rahman,"[Chennai Super Kings, Delhi Capitals]","[2023, 2024]",3/22/2024,5/1/2024
2,376116,Umesh Yadav,"[Gujarat Titans, Kolkata Knight Riders]","[2023, 2024]",3/24/2024,5/10/2024
3,390481,Harshal Patel,"[Punjab Kings, Royal Challengers Bengaluru]","[2023, 2024]",3/23/2024,5/9/2024
4,390484,Jaydev Unadkat,"[Lucknow Super Giants, Sunrisers Hyderabad]","[2023, 2024]",3/27/2024,5/8/2024
5,475281,Shardul Thakur,"[Chennai Super Kings, Kolkata Knight Riders]","[2023, 2024]",4/1/2023,5/5/2024
6,493773,Lockie Ferguson,"[Kolkata Knight Riders, Royal Challengers Beng...","[2023, 2024]",4/14/2023,5/9/2024
7,595978,Tristan Stubbs,"[Delhi Capitals, Mumbai Indians]","[2023, 2024]",4/17/2024,5/6/2023
8,625371,Hardik Pandya,"[Gujarat Titans, Mumbai Indians]","[2023, 2024]",3/24/2024,5/7/2023
9,670031,Alzarri Joseph,"[Gujarat Titans, Royal Challengers Bengaluru]","[2023, 2024]",3/22/2024,5/7/2023


In [101]:
bat_same_year = (
    df.groupby(["p_bat", "bat", "year"])["team_bat"]
      .nunique()
      .reset_index(name="team_count")
)

bat_same_year[bat_same_year["team_count"] > 1]

,p_bat,bat,year,team_count


In [102]:
bowl_same_year = (
    df.groupby(["p_bowl", "bowl", "year"])["team_bowl"]
      .nunique()
      .reset_index(name="team_count")
)

bowl_same_year[bowl_same_year["team_count"] > 1]

,p_bowl,bowl,year,team_count


In [103]:
bat_players = (
    df[["p_bat", "bat"]]
    .drop_duplicates()
    .rename(columns={"p_bat": "player_id", "bat": "bat_name"})
)

bowl_players = (
    df[["p_bowl", "bowl"]]
    .drop_duplicates()
    .rename(columns={"p_bowl": "player_id", "bowl": "bowl_name"})
)

cross_role_identity = bat_players.merge(
    bowl_players,
    on="player_id",
    how="outer"
)

cross_role_identity.head()

,player_id,bat_name,bowl_name
0,8917,Moeen Ali,Moeen Ali
1,25913,Mohammad Nabi,Mohammad Nabi
2,26421,Ravichandran Ashwin,Ravichandran Ashwin
3,28081,MS Dhoni,NaN
4,28235,Shikhar Dhawan,NaN


In [104]:
cross_role_identity[
    cross_role_identity["bat_name"].notna() &
    cross_role_identity["bowl_name"].notna() &
    (cross_role_identity["bat_name"] != cross_role_identity["bowl_name"])
]

,player_id,bat_name,bowl_name


In [105]:
cross_role_identity[
    cross_role_identity["bat_name"].isna() |
    cross_role_identity["bowl_name"].isna()
].shape

(127, 3)

In [109]:
kkr_coverage = audit_team_coverage(df)

kkr_coverage

,p_match,year,match_date,ground,winner,innings_count,opponents,complete_two_innings
0,1359476,2023,4/1/2023,"Punjab Cricket Association IS Bindra Stadium, ...",Punjab Kings,2,[Punjab Kings],True
1,1359483,2023,4/6/2023,"Eden Gardens, Kolkata",Kolkata Knight Riders,2,[Royal Challengers Bengaluru],True
2,1359487,2023,4/9/2023,"Narendra Modi Stadium, Ahmedabad",Kolkata Knight Riders,2,[Gujarat Titans],True
3,1359493,2023,4/14/2023,"Eden Gardens, Kolkata",Sunrisers Hyderabad,2,[Sunrisers Hyderabad],True
4,1359496,2023,4/16/2023,"Wankhede Stadium, Mumbai",Mumbai Indians,2,[Mumbai Indians],True
5,1359502,2023,4/20/2023,"Arun Jaitley Stadium, Delhi",Delhi Capitals,2,[Delhi Capitals],True
6,1359507,2023,4/23/2023,"Eden Gardens, Kolkata",Chennai Super Kings,2,[Chennai Super Kings],True
7,1359510,2023,4/26/2023,"M Chinnaswamy Stadium, Bengaluru",Kolkata Knight Riders,2,[Royal Challengers Bengaluru],True
8,1359513,2023,4/29/2023,"Eden Gardens, Kolkata",Gujarat Titans,2,[Gujarat Titans],True
9,1359521,2023,5/4/2023,"Rajiv Gandhi International Stadium, Uppal, Hyd...",Kolkata Knight Riders,2,[Sunrisers Hyderabad],True


In [111]:
kkr_coverage.groupby("year").size()

year
2023    14
2024    14
dtype: int64

In [112]:
kkr_coverage["opponents"].explode().value_counts()

opponents
Sunrisers Hyderabad            5
Royal Challengers Bengaluru    4
Punjab Kings                   3
Mumbai Indians                 3
Delhi Capitals                 3
Chennai Super Kings            3
Lucknow Super Giants           3
Gujarat Titans                 2
Rajasthan Royals               2
Name: count, dtype: int64

In [113]:
kkr_coverage[
    ["p_match", "year", "match_date", "ground",
     "winner", "opponents", "innings_count",
     "complete_two_innings"]
].sort_values(["year", "match_date"])

,p_match,year,match_date,ground,winner,opponents,innings_count,complete_two_innings
0,1359476,2023,4/1/2023,"Punjab Cricket Association IS Bindra Stadium, ...",Punjab Kings,[Punjab Kings],2,True
3,1359493,2023,4/14/2023,"Eden Gardens, Kolkata",Sunrisers Hyderabad,[Sunrisers Hyderabad],2,True
4,1359496,2023,4/16/2023,"Wankhede Stadium, Mumbai",Mumbai Indians,[Mumbai Indians],2,True
5,1359502,2023,4/20/2023,"Arun Jaitley Stadium, Delhi",Delhi Capitals,[Delhi Capitals],2,True
6,1359507,2023,4/23/2023,"Eden Gardens, Kolkata",Chennai Super Kings,[Chennai Super Kings],2,True
7,1359510,2023,4/26/2023,"M Chinnaswamy Stadium, Bengaluru",Kolkata Knight Riders,[Royal Challengers Bengaluru],2,True
8,1359513,2023,4/29/2023,"Eden Gardens, Kolkata",Gujarat Titans,[Gujarat Titans],2,True
1,1359483,2023,4/6/2023,"Eden Gardens, Kolkata",Kolkata Knight Riders,[Royal Challengers Bengaluru],2,True
2,1359487,2023,4/9/2023,"Narendra Modi Stadium, Ahmedabad",Kolkata Knight Riders,[Gujarat Titans],2,True
11,1359530,2023,5/11/2023,"Eden Gardens, Kolkata",Rajasthan Royals,[Rajasthan Royals],2,True


In [114]:
kkr_coverage.groupby("year").agg(
    matches=("p_match", "nunique"),
    first_match=("match_date", "min"),
    last_match=("match_date", "max"),
    complete_matches=("complete_two_innings", "sum")
)

,matches,first_match,last_match,complete_matches
year,,,,
2023,14,4/1/2023,5/8/2023,14
2024,14,3/23/2024,5/5/2024,14


In [115]:
kkr_coverage[kkr_coverage["year"] == 2024][
    ["p_match", "match_date", "opponents", "winner"]
].sort_values("match_date")

,p_match,match_date,opponents,winner
14,1422121,3/23/2024,[Sunrisers Hyderabad],Kolkata Knight Riders
15,1422128,3/29/2024,[Royal Challengers Bengaluru],Kolkata Knight Riders
18,1426266,4/14/2024,[Lucknow Super Giants],Kolkata Knight Riders
19,1426269,4/16/2024,[Rajasthan Royals],Rajasthan Royals
20,1426274,4/21/2024,[Royal Challengers Bengaluru],Kolkata Knight Riders
21,1426280,4/26/2024,[Punjab Kings],Punjab Kings
22,1426285,4/29/2024,[Delhi Capitals],Kolkata Knight Riders
16,1422134,4/3/2024,[Delhi Capitals],Kolkata Knight Riders
17,1426260,4/8/2024,[Chennai Super Kings],Chennai Super Kings
25,1426298,5/11/2024,[Mumbai Indians],Kolkata Knight Riders


In [116]:
kkr_2024_dates = pd.to_datetime(
    kkr_coverage.loc[kkr_coverage["year"] == 2024, "match_date"]
)

kkr_2024_dates.sort_values()

14   2024-03-23
15   2024-03-29
16   2024-04-03
17   2024-04-08
18   2024-04-14
19   2024-04-16
20   2024-04-21
21   2024-04-26
22   2024-04-29
23   2024-05-03
24   2024-05-05
25   2024-05-11
26   2024-05-21
27   2024-05-26
Name: match_date, dtype: datetime64[ns]

In [117]:
kkr_coverage.groupby(["year", "ground"]).size()

year  ground                                                               
2023  Arun Jaitley Stadium, Delhi                                              1
      Eden Gardens, Kolkata                                                    7
      M Chinnaswamy Stadium, Bengaluru                                         1
      MA Chidambaram Stadium, Chepauk, Chennai                                 1
      Narendra Modi Stadium, Ahmedabad                                         1
      Punjab Cricket Association IS Bindra Stadium, Mohali, Chandigarh         1
      Rajiv Gandhi International Stadium, Uppal, Hyderabad                     1
      Wankhede Stadium, Mumbai                                                 1
2024  Bharat Ratna Shri Atal Bihari Vajpayee Ekana Cricket Stadium, Lucknow    1
      Dr. Y.S. Rajasekhara Reddy ACA-VDCA Cricket Stadium, Visakhapatnam       1
      Eden Gardens, Kolkata                                                    7
      M Chinnaswamy Stadium, Beng

In [121]:
team_name_audit = audit_team_name_consistency(df)

team_name_audit

{'rcb_variants_found': ['Royal Challengers Bangalore',
  'Royal Challengers Bengaluru'],
 'variant_count': 2,
 'normalization_required': True}

In [122]:
innings_length_audit = audit_innings_length_anomalies(df)

innings_length_audit[
    innings_length_audit["max_balls"].isin([0, 90, 96])
].sort_values(["year", "match_date", "p_match", "inns"])

,p_match,inns,max_balls,delivery_count,match_date,year,max_balls_zero,reduced_length
2,1359476,1,0,123,4/1/2023,2023,True,False
145,1370353,1,0,122,5/28/2023,2023,True,False
146,1370353,2,90,94,5/28/2023,2023,False,True
265,1426298,1,96,99,5/11/2024,2024,False,True
266,1426298,2,96,102,5/11/2024,2024,False,True


In [123]:
innings_length_audit["max_balls"].value_counts().sort_index()

max_balls
0        2
90       1
96       2
120    284
Name: count, dtype: int64

In [124]:
extras_anomalies = audit_score_extras_reconciliation(df)

print("Number of reconciliation anomalies:", len(extras_anomalies))

extras_anomalies

Number of reconciliation anomalies: 14


,p_match,inns,ball_id,score,batruns,wide,noball,byes,legbyes,calculated_score,outcome
1452,1359480,2,19.02,1,0,0,1,0,1,2,leg bye
1813,1359482,1,18.04,1,0,0,1,1,0,2,bye
4565,1359493,2,9.04,1,0,0,1,0,1,2,leg bye
4951,1359495,1,10.05,1,0,0,1,0,4,5,leg bye
9151,1359512,1,15.05,1,0,0,1,0,1,2,leg bye
13024,1359528,2,12.03,1,0,0,1,0,4,5,leg bye
15180,1359537,2,18.04,1,0,0,1,4,0,5,bye
17044,1370350,1,19.04,1,0,0,1,1,0,2,bye
20870,1422131,1,5.02,1,0,0,1,0,1,2,leg bye
21786,1422134,2,14.03,1,0,0,1,4,0,5,bye


In [125]:
wicket_audit = audit_wicket_semantics(df)

wicket_audit

{'wicket_events': 1799,
 'missing_dismissal': 0,
 'missing_p_out': 0,
 'bat_out_false': 52}

In [126]:
df[
    (df["out"] == True) &
    (df["bat_out"] == False)
][
    [
        "p_match",
        "inns",
        "ball_id",
        "bat",
        "p_bat",
        "dismissal",
        "p_out",
        "bat_out",
    ]
]

,p_match,inns,ball_id,bat,p_bat,dismissal,p_out,bat_out
1760,1359482,1,10.01,Shikhar Dhawan,28235,retired not out (hurt),342619,False
2305,1359484,1,19.01,Abdul Samad,1175485,run out,1246528,False
4257,1359492,1,19.07,Harpreet Brar,1168641,run out,290727,False
5399,1359497,1,4.06,Shubman Gill,1070173,run out,1151288,False
5989,1359499,1,19.06,Cameron Green,1076713,run out,892749,False
6237,1359500,1,19.05,Krunal Pandya,471342,run out,604302,False
6316,1359500,2,12.04,Jos Buttler,308967,run out,425943,False
6521,1359501,2,5.04,Prabhsimran Singh,1161024,run out,340854,False
6722,1359502,1,19.06,Andre Russell,276298,run out,1108375,False
6964,1359503,1,19.07,Marco Jansen,696401,run out,719715,False


In [127]:
tactical_missingness = audit_tactical_missingness(df)

tactical_missingness

,column,missing,missing_pct
0,bowl_speed_category,2379,6.803558
1,bowl_release_point,2379,6.803558
2,bowl_type,1936,5.536649
3,control,963,2.754025
4,shot,194,0.554809
5,line,56,0.160151
6,length,47,0.134412


In [128]:
tactical_missingness.sort_values(
    "missing",
    ascending=False
)

,column,missing,missing_pct
0,bowl_speed_category,2379,6.803558
1,bowl_release_point,2379,6.803558
2,bowl_type,1936,5.536649
3,control,963,2.754025
4,shot,194,0.554809
5,line,56,0.160151
6,length,47,0.134412


In [129]:
delivery_key_audit = audit_delivery_key(df)

delivery_key_audit

{'key_columns': ['p_match', 'inns', 'ball_id'],
 'duplicate_key_rows': 0,
 'unique_key': np.True_}

In [130]:
incomplete_matches = audit_incomplete_matches(df)

incomplete_matches

,p_match,innings_count,winner,year,match_date
44,1359519,1,-,2023,5/3/2023


In [131]:
schema_naming = audit_schema_naming()

schema_naming

,data_dictionary_name,csv_name
0,over,over_num
1,date,match_date
2,wagonX,wagonx
3,wagonY,wagony
4,wagonZone,wagonzone


In [132]:
anomaly_summary = pd.DataFrame([
    {
        "finding": "RCB naming variants",
        "count": team_name_audit["variant_count"],
        "severity": "Medium",
        "classification": "Standardization",
        "action": "Normalize in derived analytical data",
    },
    {
        "finding": "max_balls = 0 innings",
        "count": int(
            innings_length_audit["max_balls_zero"].sum()
        ),
        "severity": "Medium",
        "classification": "Metadata anomaly",
        "action": "Flag; do not interpret literally",
    },
    {
        "finding": "Reduced-length innings",
        "count": int(
            innings_length_audit["reduced_length"].sum()
        ),
        "severity": "Informational",
        "classification": "Valid metadata",
        "action": "Preserve",
    },
    {
        "finding": "Score/extras reconciliation",
        "count": len(extras_anomalies),
        "severity": "Low",
        "classification": "Component overlap",
        "action": "Preserve raw values",
    },
    {
        "finding": "Wicket events",
        "count": wicket_audit["wicket_events"],
        "severity": "Informational",
        "classification": "Cricket semantics",
        "action": "Use out=True",
    },
    {
        "finding": "Wickets with bat_out=False",
        "count": wicket_audit["bat_out_false"],
        "severity": "Informational",
        "classification": "Cricket semantics",
        "action": "Do not use bat_out as wicket counter",
    },
    {
        "finding": "Incomplete matches",
        "count": len(incomplete_matches),
        "severity": "Informational",
        "classification": "Coverage",
        "action": "Exclude only from completed-match analyses",
    },
    {
        "finding": "Duplicate delivery keys",
        "count": delivery_key_audit["duplicate_key_rows"],
        "severity": "Critical",
        "classification": "Key integrity",
        "action": "Must remain zero",
    },
])

anomaly_summary

,finding,count,severity,classification,action
0,RCB naming variants,2,Medium,Standardization,Normalize in derived analytical data
1,max_balls = 0 innings,2,Medium,Metadata anomaly,Flag; do not interpret literally
2,Reduced-length innings,3,Informational,Valid metadata,Preserve
3,Score/extras reconciliation,14,Low,Component overlap,Preserve raw values
4,Wicket events,1799,Informational,Cricket semantics,Use out=True
5,Wickets with bat_out=False,52,Informational,Cricket semantics,Do not use bat_out as wicket counter
6,Incomplete matches,1,Informational,Coverage,Exclude only from completed-match analyses
7,Duplicate delivery keys,0,Critical,Key integrity,Must remain zero


In [135]:
delivery_order_audit = audit_delivery_order(df)

delivery_order_audit

,p_match,physical_innings_order,expected_innings_order,order_mismatch
11,1359486,"[2, 1]","[1, 2]",True


In [136]:
print(
    "Matches with physical innings-order mismatch:",
    len(delivery_order_audit)
)

Matches with physical innings-order mismatch: 1
